# Контроль качества данных эксперимента 3

**Статус:** техническая инвентаризация и диагностический ноутбук
реокардиомонитора РНЦХ. Функции сердца здесь не рассчитываются.

Паспорт серии приведён в
[`10.10`](10.10_Паспорт_эксперимента_3.md). Дата остаётся открытым полем до
сверки с первичным протоколом. Постороннее исследование на другом приборе не
включается: ноутбук обрабатывает только записи, явно перечисленные во внешней
конфигурации.

Численные наблюдения перепроверены 27.08.2026 по четырём основным CSV из
локальной конфигурации и их полным контрольным суммам. Сохранённые выводы ячеек очищены. Обезличенная подробная сводка хранится
во внешнем `derived_root` в
`exp03/qc/10.11_primary_signal_diagnostic.candidate.json` со статусом
`pending_manual_review`.

Исторические дыхательные интервалы используются только как гипотезы для
повторного просмотра. В `11.11` сохранены кандидатные интервалы `14-09-42` и
`14-19-52`; для `14-07-01` полный кандидат не обоснован, а `14-17-16`
исключена как технический тест. Командных меток нет, поэтому ни один кандидат
не становится количественной разметкой до явного решения автора, основанного на памяти о протоколе и форме
сигнала.


In [ ]:
# Импорты и внешний контракт данных
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CONFIG_ENV = "KALMYKOV_EXP03_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp03_paths.example.json"
    )

CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
CSV_ROOT = (DATA_ROOT / CONFIG["csv_subdir"]).resolve()
CSV_ROOT.relative_to(DATA_ROOT)

RECORDINGS = CONFIG["recordings"]
record_ids = [item["record_id"] for item in RECORDINGS]
relative_paths = [item["relative_path"] for item in RECORDINGS]
if not RECORDINGS or len(record_ids) != len(set(record_ids)):
    raise ValueError("recordings должен содержать уникальные непустые record_id")
if len(relative_paths) != len(set(relative_paths)):
    raise ValueError("Один relative_path нельзя назначать нескольким record_id")

SOURCE_COLUMNS = CONFIG["source_columns"]
CANONICAL_COLUMNS = [
    "time_s", "rheo_1_mohm", "base_1_ohm", "qs_1_ohm",
    "ecg_v", "rheo_2_mohm", "base_2_ohm", "qs_2_ohm",
]
if len(SOURCE_COLUMNS) != len(CANONICAL_COLUMNS):
    raise ValueError("source_columns должен описывать ровно восемь столбцов CSV")

ACTIVE_THRESHOLD_OHM = float(CONFIG["active_channel_threshold_ohm"])
if not np.isfinite(ACTIVE_THRESHOLD_OHM) or ACTIVE_THRESHOLD_OHM <= 0:
    raise ValueError("active_channel_threshold_ohm должен быть положительным")

plt.rcParams.update({
    "figure.figsize": (15, 9),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 13,
})


def resolve_record_path(relative_path):
    path = (DATA_ROOT / relative_path).resolve()
    path.relative_to(CSV_ROOT)
    if not path.is_file():
        raise FileNotFoundError(f"Нет файла для записи: {relative_path}")
    return path


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_record(path):
    frame = pd.read_csv(path)
    if list(frame.columns) != SOURCE_COLUMNS:
        raise ValueError(
            "Схема CSV не совпадает с source_columns внешней конфигурации"
        )
    frame.columns = CANONICAL_COLUMNS
    frame = frame.apply(pd.to_numeric, errors="raise")
    values = frame.to_numpy(dtype=float)
    if len(frame) < 2 or not np.isfinite(values).all():
        raise ValueError("CSV пуст, слишком короток или содержит нечисловые значения")
    dt = np.diff(frame["time_s"].to_numpy(dtype=float))
    if not np.all(dt > 0):
        raise ValueError("TIME должен строго возрастать")
    return frame


## 1. Инвентаризация и соответствие файлов

В анализ входят только обезличенные `record_id`, явно перечисленные в
`KALMYKOV_EXP03_CONFIG`. Относительный путь обязан находиться внутри
разрешённого `csv_subdir`. Остальные CSV учитываются только числом при проверке
полноты конфигурации и не анализируются.

Полный SHA-256 определяет байт-в-байт совпадающие записи. Канонический файл
задаётся конфигурацией, а не выбирается автоматически по длине пути.


In [ ]:
# Инвентаризация разрешённой области и явно включённых записей
discovered = sorted(CSV_ROOT.rglob("*.csv"))
discovered_hashes = {path: file_sha256(path) for path in discovered}
copy_count_by_hash = pd.Series(list(discovered_hashes.values())).value_counts()

inventory_rows = []
listed_paths = set()
for spec in RECORDINGS:
    path = resolve_record_path(spec["relative_path"])
    listed_paths.add(path)
    data = read_record(path)
    dt = np.diff(data["time_s"].to_numpy(dtype=float))
    sha256 = discovered_hashes.get(path, file_sha256(path))
    inventory_rows.append({
        "record_id": spec["record_id"],
        "path": path,
        "sha256": sha256,
        "sha256_prefix": sha256[:16],
        "copies_in_allowed_root": int(copy_count_by_hash.get(sha256, 1)),
        "rows": len(data),
        "duration_s": float(data["time_s"].iloc[-1] - data["time_s"].iloc[0]),
        "fs_hz": float(1.0 / np.median(dt)),
        "role": spec.get("role", "не указана"),
        "protocol_reference": spec.get("protocol_reference", "не указана"),
        "expected_active_channels": spec.get("expected_active_channels", []),
    })

inventory = pd.DataFrame(inventory_rows)
inventory["duplicate_in_list"] = inventory.duplicated("sha256", keep=False)
if inventory["duplicate_in_list"].any():
    duplicate_ids = inventory.loc[inventory["duplicate_in_list"], "record_id"].tolist()
    raise ValueError(f"В recordings перечислены дублирующие записи: {duplicate_ids}")

unlisted_count = len(set(discovered) - listed_paths)
mapping_table = inventory[[
    "record_id", "role", "protocol_reference", "expected_active_channels",
    "duration_s", "fs_hz", "copies_in_allowed_root", "sha256_prefix",
]].copy()
mapping_table["duration_s"] = mapping_table["duration_s"].round(2)
mapping_table["fs_hz"] = mapping_table["fs_hz"].round(2)
mapping_table.columns = [
    "Запись", "Роль", "Ссылка на протокол", "Ожидаемые активные каналы",
    "Длительность, с", "Частота, Гц", "Копий в разрешённой области",
    "SHA-256, начало",
]
display(mapping_table.style.hide(axis="index").set_properties(**{"text-align": "left"}))
print(f"Явно включено записей: {len(inventory)}")
print(f"Других CSV в разрешённой области: {unlisted_count}")
print("Неуказанные CSV не анализируются.")


### Результат файловой проверки

- запись `14-07-01` соответствует отдельному каналу 1, ТТРКГ;
- запись `14-09-42` соответствует отдельному каналу 2, боковой сборке;
- запись `14-17-16` содержит поочерёдное отключение каналов, тогда как в
  текстовом протоколе для этой пробы указано 14:08;
- запись `14-19-52` содержит совместную дыхательную регистрацию, тогда как в
  текстовом протоколе указано 14:17;
- четыре основные записи имели байт-в-байт копии в двух каталогах разрешённой
  области данных.

Повторная проверка подтвердила соответствие назначенных ролей фактическим
состояниям каналов по техническому порогу `BASE > 5 Ом`. Установленное
несоответствие двух временных обозначений не объясняется. Соответствие записей
протокольным пробам можно уточнять по первичному журналу, но временных меток
дыхательных команд нет ни в журнале данных, ни в CSV. Последовательность и
границы режимов принимаются только после авторской реконструкции по памяти и
форме сигнала. Дата эксперимента по
этим временам не определяется.


In [ ]:
record_by_id = dict(zip(inventory["record_id"], inventory["path"]))
spec_by_id = {item["record_id"]: item for item in RECORDINGS}


def path_for_role(role):
    matches = [item for item in RECORDINGS if item.get("role") == role]
    if len(matches) != 1:
        raise ValueError(f"Для роли {role!r} ожидается ровно одна запись")
    return record_by_id[matches[0]["record_id"]], matches[0]["record_id"]


def channel_stats(path, channels):
    data = read_record(path)
    result = []
    for channel in channels:
        base = data[f"base_{channel}_ohm"]
        qs = data[f"qs_{channel}_ohm"]
        rheo = data[f"rheo_{channel}_mohm"]
        result.append({
            "Канал": channel,
            "BASE median, Ом": base.median(),
            "BASE p01–p99, Ом": f"{base.quantile(0.01):.2f}–{base.quantile(0.99):.2f}",
            "QS median, Ом": qs.median(),
            "QS max, Ом": qs.max(),
            "RHEO p01–p99, мОм": f"{rheo.quantile(0.01):.1f}–{rheo.quantile(0.99):.1f}",
        })
    table = pd.DataFrame(result)
    for column in ["BASE median, Ом", "QS median, Ом", "QS max, Ом"]:
        table[column] = table[column].round(2)
    return table


def plot_record(path, title, channels):
    data = read_record(path)
    time = data["time_s"]
    fs = 1.0 / np.median(np.diff(time))
    window = max(1, int(round(fs)))
    if window % 2 == 0:
        window += 1

    fig, axes = plt.subplots(
        4, 1, figsize=(15, 10), sharex=True,
        gridspec_kw={"height_ratios": [2.2, 1, 1, 1]},
    )
    colors = {1: "#2563eb", 2: "#dc2626"}
    for channel in channels:
        rheo = data[f"rheo_{channel}_mohm"]
        smooth = rheo.rolling(window, center=True, min_periods=1).median()
        axes[0].plot(time, rheo, color=colors[channel], alpha=0.16, linewidth=0.5)
        axes[0].plot(
            time, smooth, color=colors[channel], linewidth=1.7,
            label=f"канал {channel}, медиана 1 с",
        )
        axes[1].plot(
            time, data[f"base_{channel}_ohm"], color=colors[channel],
            linewidth=1.2, label=f"BASE {channel}",
        )
        axes[2].plot(
            time, data[f"qs_{channel}_ohm"], color=colors[channel],
            linewidth=1.2, label=f"QS {channel}",
        )

    axes[3].plot(time, data["ecg_v"], color="#111827", linewidth=0.7, label="ЭКГ")
    axes[0].set_ylabel("RHEO, мОм")
    axes[1].set_ylabel("BASE, Ом")
    axes[2].set_ylabel("QS, Ом")
    axes[3].set_ylabel("ЭКГ, В")
    axes[3].set_xlabel("Время от начала CSV, с")
    axes[0].set_title(title)
    for axis in axes:
        axis.legend(loc="upper right", ncol=max(1, len(channels)))
        axis.margins(x=0)
    fig.tight_layout()
    plt.show()


def active_intervals(path, threshold_ohm=ACTIVE_THRESHOLD_OHM):
    data = read_record(path)
    time = data["time_s"].to_numpy()
    base_1 = data["base_1_ohm"].to_numpy()
    base_2 = data["base_2_ohm"].to_numpy()
    state = (base_1 > threshold_ohm).astype(int) + 2 * (base_2 > threshold_ohm).astype(int)
    cuts = np.r_[0, np.flatnonzero(state[1:] != state[:-1]) + 1, len(state)]
    names = {0: "ни один", 1: "только 1", 2: "только 2", 3: "оба"}
    rows = []
    for left, right in zip(cuts[:-1], cuts[1:]):
        if time[right - 1] - time[left] < 0.25:
            continue
        rows.append({
            "Начало, с": time[left],
            "Конец, с": time[right - 1],
            "Активны по порогу": names[int(state[left])],
            "BASE1 median, Ом": np.median(base_1[left:right]),
            "BASE2 median, Ом": np.median(base_2[left:right]),
            "QS1 median, Ом": np.median(data["qs_1_ohm"].iloc[left:right]),
            "QS2 median, Ом": np.median(data["qs_2_ohm"].iloc[left:right]),
        })
    return pd.DataFrame(rows).round(2)



# Диагностический критерий длительных плоских участков у наблюдаемых пределов.
# Порог не является паспортной границей прибора и не доказывает насыщение.
EXTREME_ENDPOINT_BAND_FRACTION = 0.01
MIN_EXTREME_PLATEAU_DURATION_S = 0.10


def extreme_flat_plateau_mask(
    data,
    channel,
    endpoint_band_fraction=EXTREME_ENDPOINT_BAND_FRACTION,
    minimum_duration_s=MIN_EXTREME_PLATEAU_DURATION_S,
):
    time = data["time_s"].to_numpy(dtype=float)
    values = data[f"rheo_{channel}_mohm"].to_numpy(dtype=float)
    dt = float(np.median(np.diff(time)))
    value_range = float(values.max() - values.min())
    lower_bound = float(values.min() + endpoint_band_fraction * value_range)
    upper_bound = float(values.max() - endpoint_band_fraction * value_range)

    boundaries = np.flatnonzero(np.r_[True, values[1:] != values[:-1], True])
    mask = np.zeros(len(values), dtype=bool)
    rows = []
    for left, right in zip(boundaries[:-1], boundaries[1:]):
        duration_s = float((right - left) * dt)
        value = float(values[left])
        endpoint = None
        if value <= lower_bound:
            endpoint = "нижняя"
        elif value >= upper_bound:
            endpoint = "верхняя"
        if endpoint is None or duration_s < minimum_duration_s:
            continue
        mask[left:right] = True
        rows.append({
            "Канал": channel,
            "Начало, с": time[left],
            "Конец, с": time[right - 1] + dt,
            "Длительность, с": duration_s,
            "Уровень RHEO, мОм": value,
            "Граница диапазона": endpoint,
        })

    columns = [
        "Канал", "Начало, с", "Конец, с", "Длительность, с",
        "Уровень RHEO, мОм", "Граница диапазона",
    ]
    return mask, pd.DataFrame(rows, columns=columns).round(3)


def extreme_plateau_fraction_by_interval(data, channels, intervals):
    time = data["time_s"].to_numpy(dtype=float)
    masks = {
        channel: extreme_flat_plateau_mask(data, channel)[0]
        for channel in channels
    }
    rows = []
    for label, start_s, stop_s in intervals:
        interval_mask = (time >= start_s) & (time < stop_s)
        if not interval_mask.any():
            raise ValueError(f"Пустой диагностический интервал: {label}")
        for channel in channels:
            fraction = float(masks[channel][interval_mask].mean())
            if fraction > 0:
                amplitude_status = (
                    "амплитуда полного интервала не принимается; "
                    "требуется чистое подокно"
                )
            else:
                amplitude_status = (
                    "этот критерий не блокирует; остальные проверки обязательны"
                )
            rows.append({
                "Интервал-кандидат": label,
                "Начало, с": start_s,
                "Конец, с": stop_s,
                "Канал": channel,
                "Плоские крайние отсчёты, %": 100.0 * fraction,
                "Статус амплитуды": amplitude_status,
            })
    return pd.DataFrame(rows).round({
        "Начало, с": 3,
        "Конец, с": 3,
        "Плоские крайние отсчёты, %": 2,
    })


## 2. Отдельная запись ТТРКГ, канал 1

Назначение файла и работа только канала 1 подтверждены. Исторические границы
0, 30, 45, 53 и 67,42 с повторно сопоставлены с `BASE1`. Форма сигнала
поддерживает лишь общий переход к более нестационарному участку после 30 с и
восстановление около 53–54 с; границы 45 и 53 с не имеют самостоятельного
надёжного подтверждения. Полная кандидатная последовательность дыхательных
режимов не формируется, `mode_sequence` остаётся пустой. Это отрицательное
решение зафиксировано в `11.11` как отсутствие обоснованного размеченного
кандидата.


In [ ]:
path, record_id = path_for_role("ttrkg_channel_1_only")
display(channel_stats(path, [1]).style.hide(axis="index"))
plot_record(path, f"{record_id}: ТТРКГ, канал 1 отдельно", [1])


**Диагностическое наблюдение.** Медиана `BASE1` равна 101,353 Ом, тогда
как в примечании протокола указано около 80 Ом. `QS1` сохраняет значение
4700 Ом на протяжении всей записи. Физический смысл этого кода и статус
предела шкалы не установлены.

В `RHEO1` повторяются уровни около −584,501 и +584,215 мОм. Доля наиболее
частых дискретных крайних уровней составляет 3,26 % отсчётов, а самый длинный
непрерывный участок — 0,57 с. Это наблюдение о записанных числах. Насыщение
измерительного тракта остаётся гипотезой; запись требует расшифровки `QS` и
проверки диапазона `RHEO`.


## 3. Отдельная запись боковой сборки, канал 2

Назначение файла и работа только канала 2 подтверждены. Повторный просмотр
`BASE2` поддерживает кандидатные границы 0, 30, 49, 59 и 69,22 с. В `11.11`
им предварительно сопоставлены спокойное дыхание, вдох с задержкой,
форсированное дыхание и задержка после выдоха. Физиологические названия и
точные секунды не зарегистрированы прибором, поэтому разметка имеет статус
кандидата, ожидающего ручного принятия. До принятия её нельзя передавать в
количественный анализ.


In [ ]:
path, record_id = path_for_role("side_channel_2_only")
display(channel_stats(path, [2]).style.hide(axis="index"))
plot_record(path, f"{record_id}: боковая сборка, канал 2 отдельно", [2])


**Диагностическое наблюдение.** Медиана `BASE2` равна 37,158 Ом и близка
к значению 36 Ом из протокола. `QS2` находится около 343 Ом и не принимает
значение 4700 Ом. `BASE1` равен нулю, что согласуется с отключением канала 1.
Близость базового уровня протоколу не заменяет калибровку прибора.

В `RHEO2` уровни около −505,679 и +505,432 мОм занимают 12,44 % отсчётов;
самый длинный непрерывный участок длится 2,31 с. До проверки измерительного
тракта эти участки нельзя использовать для оценки дыхательной или пульсовой
амплитуды. Причина повторения крайних уровней не установлена.


## 4. Запись поочерёдного отключения каналов

Кандидатные состояния определяются по порогу `BASE`, заданному во внешней
конфигурации. Порог является техническим правилом сегментации, а не
физическим критерием исправности. Переходы и подписи состояний должны быть
проверены по протоколу до использования численных различий.

Запись является техническим тестом и исключена из дыхательной разметки
серии `11.11`: переходы `BASE` отражают прежде всего изменение состояния
каналов.


In [ ]:
path, record_id = path_for_role("channel_switch_test")
switch_intervals = active_intervals(path)
display(switch_intervals.style.hide(axis="index"))
plot_record(path, f"{record_id}: поочерёдное отключение каналов", [1, 2])


**Диагностическое наблюдение.** Технические состояния по `BASE > 5 Ом`
образуют последовательность: оба канала 0,000–12,075 с; только канал 2
12,080–22,990 с; оба канала 22,995–34,025 с; только канал 1
34,030–42,805 с; оба канала 42,810–46,020 с. В последнем интервале длительностью
3,21 с оценка менее устойчива и не проходит установленный минимум 4 с.

При совместной работе медианы составляют приблизительно `BASE1=54,2 Ом` и
`BASE2=40,9 Ом`. После отключения канала 1 `BASE2` остаётся около 36,9 Ом;
после отключения канала 2 `BASE1` возрастает примерно до 92,7 Ом. Зависимость
базового уровня оставшегося канала от состояния соседнего подтверждается без
ЭКГ-разметки. Причина эффекта не определена.

Повторяющиеся крайние уровни занимают 3,45 % отсчётов `RHEO1` и 5,98 %
отсчётов `RHEO2`; непрерывные участки достигают 0,91 и 1,03 с. Поэтому
пульсовые амплитуды внутри этих интервалов пока не являются принятым
результатом. Значение `QS=4700` встречается у отключённого канала и у канала 1
при одиночной работе, однако его физическая интерпретация неизвестна.


## 5. Совместная запись обоих каналов с дыхательными манёврами

Работа обоих каналов на всей записи подтверждена. В `11.11` сохранена
кандидатная последовательность: 0–30 с — спокойное дыхание; 30–46 с — вдох и
задержка; 46–57 с — форсированное дыхание; 57–73,62 с — задержка после выдоха.
Границы перенесены из исторического ноутбука 16 и повторно сопоставлены с
`BASE1` и `BASE2`, но событийных меток команд нет. Поэтому эта разметка ожидает
ручного принятия и не является непосредственно измеренной шкалой режимов.


In [ ]:
path, record_id = path_for_role("both_channels_breathing")
data_both = read_record(path)
display(channel_stats(path, [1, 2]).style.hide(axis="index"))
plot_record(path, f"{record_id}: оба канала, дыхательный протокол", [1, 2])

plateau_tables = []
for channel in (1, 2):
    _, channel_plateaus = extreme_flat_plateau_mask(data_both, channel)
    plateau_tables.append(channel_plateaus)
plateau_runs = pd.concat(plateau_tables, ignore_index=True)
display(plateau_runs.style.hide(axis="index"))

record_dt = float(np.median(np.diff(data_both["time_s"])))
candidate_intervals = [
    ("спокойное дыхание", 0.0, 30.0),
    ("вдох и задержка", 30.0, 46.0),
    ("форсированное дыхание", 46.0, 57.0),
    (
        "задержка после выдоха",
        57.0,
        float(data_both["time_s"].iloc[-1] + record_dt / 2.0),
    ),
]
interval_plateau_qc = extreme_plateau_fraction_by_interval(
    data_both,
    [1, 2],
    candidate_intervals,
)
display(interval_plateau_qc.style.hide(axis="index"))


mode_ids = [
    "quiet_breathing_candidate",
    "inspiration_and_hold_candidate",
    "forced_breathing_candidate",
    "expiration_hold_candidate",
]
plateau_masks = {
    channel: extreme_flat_plateau_mask(data_both, channel)[0]
    for channel in (1, 2)
}
plateau_records = []
for row in plateau_runs.to_dict(orient="records"):
    plateau_records.append({
        "channel": int(row["Канал"]),
        "start_s": float(row["Начало, с"]),
        "stop_s": float(row["Конец, с"]),
        "duration_s": float(row["Длительность, с"]),
        "rheo_level_mohm": float(row["Уровень RHEO, мОм"]),
        "observed_endpoint": "lower" if row["Граница диапазона"] == "нижняя" else "upper",
    })

interval_records = []
record_stop_s = float(data_both["time_s"].iloc[-1])
for mode_id, (label, start_s, stop_s) in zip(mode_ids, candidate_intervals):
    selection = (
        (data_both["time_s"].to_numpy(dtype=float) >= start_s)
        & (data_both["time_s"].to_numpy(dtype=float) < stop_s)
    )
    channels = []
    for channel in (1, 2):
        fraction = float(plateau_masks[channel][selection].mean())
        channels.append({
            "channel": channel,
            "extreme_flat_plateau_fraction": fraction,
            "full_interval_amplitude_status": (
                "blocked_requires_clean_subwindow"
                if fraction > 0
                else "not_blocked_by_this_criterion_other_qc_required"
            ),
        })
    interval_records.append({
        "mode_id": mode_id,
        "candidate_label_ru": label,
        "start_s": float(start_s),
        "stop_s": min(float(stop_s), record_stop_s),
        "channels": channels,
    })

signal_mask_artifact = {
    "schema_version": 1,
    "artifact_type": "exp03_record_signal_exclusion_mask",
    "analysis_version": "exp03-extreme-flat-plateau-v1",
    "status": "pending_manual_review",
    "record_id": record_id,
    "input": {
        "sha256": file_sha256(path),
        "duration_s": record_stop_s - float(data_both["time_s"].iloc[0]),
        "sampling_frequency_hz": 1.0 / record_dt,
    },
    "methods": {
        "active_channel_threshold_ohm": ACTIVE_THRESHOLD_OHM,
        "extreme_endpoint_band_fraction": EXTREME_ENDPOINT_BAND_FRACTION,
        "minimum_exact_plateau_duration_s": MIN_EXTREME_PLATEAU_DURATION_S,
        "interpretation": (
            "recorded_flat_extreme_fragment; hardware_saturation_not_proven"
        ),
    },
    "active_channels": [
        {
            "channel": channel,
            "base_above_threshold_fraction": float(
                (data_both[f"base_{channel}_ohm"] > ACTIVE_THRESHOLD_OHM).mean()
            ),
            "qs_min_ohm": float(data_both[f"qs_{channel}_ohm"].min()),
            "qs_median_ohm": float(data_both[f"qs_{channel}_ohm"].median()),
            "qs_max_ohm": float(data_both[f"qs_{channel}_ohm"].max()),
            "qs_4700_fraction": float(
                (data_both[f"qs_{channel}_ohm"] == 4700.0).mean()
            ),
            "rheo_min_mohm": float(data_both[f"rheo_{channel}_mohm"].min()),
            "rheo_max_mohm": float(data_both[f"rheo_{channel}_mohm"].max()),
            "extreme_flat_plateau_fraction": float(plateau_masks[channel].mean()),
        }
        for channel in (1, 2)
    ],
    "extreme_flat_plateau_runs": plateau_records,
    "candidate_interval_qc": interval_records,
    "ecg_dependency": {
        "owner": "11.12",
        "status": "candidate_not_manually_accepted",
        "usable_for_cycle_segmentation": False,
    },
    "limitations": [
        "candidate breathing intervals are retrospective expert reconstruction",
        "absence of detected plateaus does not establish signal validity",
        "hardware saturation, QS meaning, sign and calibration are not established",
        "samples inside excluded runs must not be reconstructed by interpolation",
    ],
}

EXP03_SIGNAL_MASK_PATH = (
    DERIVED_ROOT
    / "exp03"
    / "qc"
    / "10.11_exp03_both_breathing_signal_mask.candidate.json"
)
EXP03_SIGNAL_MASK_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_mask_path = EXP03_SIGNAL_MASK_PATH.with_suffix(".json.tmp")
temporary_mask_path.write_text(
    json.dumps(signal_mask_artifact, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
temporary_mask_path.replace(EXP03_SIGNAL_MASK_PATH)
print("Кандидатная маска исключения:", EXP03_SIGNAL_MASK_PATH)


**Диагностическое наблюдение.** Оба канала активны по техническому порогу
`BASE > 5 Ом` на протяжении всей записи. Медианы равны `BASE1=54,419 Ом` и
`BASE2=40,717 Ом`; второй уровень близок к протокольному, первый ниже указанного
значения 60 Ом. `QS1` изменяется от 1272 до 1584 Ом при медиане 1307 Ом, а
`QS2` — от 362 до 372 Ом при медиане 368 Ом. Код 4700 Ом в этой записи не
встречается. Физическая интерпретация `QS` и метрологический статус шкал всё
равно остаются неизвестными.

Для воспроизводимого поиска подозрительных участков введён диагностический
критерий: значение `RHEO` должно оставаться в точности постоянным не менее
0,10 с и находиться в однопроцентной полосе у одного из двух наблюдаемых
пределов диапазона записи. Порог 1 % и длительность 0,10 с являются правилами
диагностического контроля качества, а не паспортными характеристиками
прибора. Обнаружение такого участка
подтверждает плоский крайний фрагмент в CSV, но само по себе не доказывает
насыщение измерительного тракта.

По этому критерию плоские крайние фрагменты занимают 6,32 % отсчётов `RHEO1`
и 12,54 % отсчётов `RHEO2`; их максимальная непрерывная длительность равна
соответственно 1,01 и 1,89 с. Ранее сохранённые во внешней диагностической сводке версии 1
значения 6,88 и 12,58 % получены другим правилом — суммированием частот
нескольких выбранных дискретных уровней. Эти две оценки нельзя смешивать;
после принятия критерия эту сводку требуется пересчитать.

Сопоставление с кандидатной разметкой `11.11` даёт следующую картину:

- на участке 0–30 с критерий не обнаруживает плоских крайних фрагментов ни в
  одном канале;
- на участке 30–46 с они занимают 7,06 % `RHEO1` и 12,19 % `RHEO2`; основная
  группа приходится примерно на 31,4–33,4 с;
- на участке 46–57 с доли возрастают до 32,00 и 66,23 %, поэтому амплитуда
  полного интервала не подлежит количественной интерпретации;
- на участке 57–73,62 с критерий снова не обнаруживает таких фрагментов.

Отсутствие плоских крайних участков не означает автоматическую пригодность
сигнала. Для интервалов 0–30 и 57–73,62 с всё ещё нужны принятая дыхательная и
ЭКГ-разметка, калибровка и проверка воспроизводимости. Для 30–46 с допускается
искать отдельное чистое подокно после 33,4 с, но нельзя вычислять амплитуду по
полному интервалу. Фрагменты с плоскими крайними уровнями нельзя исправлять
интерполяцией: истинная форма сигнала внутри них неизвестна.

При выполнении ноутбук сохраняет обезличенную интервальную маску и параметры
критерия во внешнем производном файле
`exp03/qc/10.11_exp03_both_breathing_signal_mask.candidate.json`. Его статус
остаётся `pending_manual_review`; файл не разрешает количественный анализ
автоматически.


## 6. Диагностическая проверка автоматических ЭКГ-кандидатов

Для сопоставления с архивным расчётом воспроизведён детектор текущей серии
`11.12` с параметрами локальной конфигурации. Кандидатный набор характеризуют
число событий, частота по числу событий, частота по медиане RR и максимальный
интервал RR. Эти показатели не являются принятой ЭКГ-разметкой.

| Запись | Кандидаты R | Частота по числу, уд/мин | Частота по медиане RR, уд/мин | Максимальный RR, с |
|---|---:|---:|---:|---:|
| `14-07-01` | 71 | 63,2 | 76,9 | 3,62 |
| `14-09-42` | 24 | 20,8 | 49,6 | 10,46 |
| `14-17-16` | 4 | 5,2 | 6,0 | 11,96 |
| `14-19-52` | 80 | 65,2 | 75,0 | 5,66 |

Во всех наборах имеются длинные пропуски. Для записи `14-19-52` алгоритм
выбрал отрицательные отклонения ЭКГ и сформировал 80 кандидатов. Это
вычислительная полярность, а не подтверждение физиологической природы
каждого события. В кандидатном списке остаются интервалы без событий
37,54–39,36; 39,36–45,02 и 45,02–46,54 с; центральный пропуск длится
5,66 с. Следовательно, этот список неполон и пока непригоден для разбиения
`RHEO` на сердечные циклы. Сам канал ЭКГ на этом основании не объявляется
непригодным: требуется ручной просмотр исходной формы в серии `11.12`.

В записи `14-17-16` четыре автоматических события находятся не далее
0,57 с от границ технических состояний каналов, поэтому их нельзя принимать
как R-зубцы без ручной проверки. Ни один сопроводительный файл ЭКГ не имеет
статуса `accepted`.

Архивный детектор `40.90` воспроизводит прежние количества 72, 72, 40 и 70
событий, но максимальные RR остаются равными 7,18; 5,26; 7,59 и 14,84 с.
Следовательно, старый критерий по числу событий и медианной частоте пропускал
длинные участки без надёжной детекции. Старые ансамблевые амплитуды и вывод о
сердечном происхождении не подтверждены.


## 7. Выводы и границы результата

### Подтверждённые файловые и экспериментальные факты

1. В разрешённой области данных обнаружены байт-в-байт копии четырёх основных
   записей; расчёт должен использовать каждый полный SHA-256 один раз.
2. Назначенные роли четырёх основных записей совпадают с фактическими
   состояниями каналов по техническому порогу `BASE > 5 Ом`.
3. В записи поочерёдного отключения базовый уровень оставшегося канала зависит
   от состояния другого канала. Механизм эффекта не установлен.
4. Временные обозначения двух записей не совпадают с текстовым протоколом.
5. Постороннее несинхронное исследование на другом приборе не относится к
   эксперименту 3.

### Диагностические наблюдения

- во всех активных `RHEO` основных записей имеются продолжительные повторения
  дискретных крайних уровней; для `14-19-52` их интервальная локализация
  воспроизведена отдельным критерием, а насыщение остаётся гипотезой;
- в `14-19-52` интервал 46–57 с нельзя использовать для количественной
  амплитуды обоих каналов, полный интервал 30–46 с также требует выделения
  чистого подокна; отсутствие найденных плато на 0–30 и 57–73,62 с не
  заменяет остальных проверок;
- автоматические ЭКГ-кандидаты текущего и архивного детекторов содержат
  длинные пропуски и не приняты вручную;
- отдельная запись канала 2 имеет `BASE2`, близкий к протокольному значению, и
  стабильный `QS2` без значения 4700 Ом;
- отдельная запись канала 1 имеет `BASE1` выше протокольного значения и
  постоянный код `QS1=4700 Ом`; физический смысл этого кода не установлен.

### Требования к продолжению

1. Сверить дату и назначение файлов по первичному журналу. По текущему
   решению автора кандидатные интервалы `11.11` остаются в статусе
   `pending_manual_review`; `accepted_modes` не заполняются и количественные
   расчёты по режимам не выполняются.
2. Проверить `QS`, пределы диапазона, калибровку `BASE/RHEO` и межканальное
   влияние по отдельному аппаратному протоколу.
3. Выполнить ручную ЭКГ-разметку с явным контролем пропусков, ложных событий и
   участков переключения каналов.
4. Только после принятия сопроводительных файлов дыхания и ЭКГ повторить
   пульсовые ансамбли и сравнение конфигураций. Числа архивного `40.90` до
   этого не использовать.
5. Числа версии РНЦХ не использовать как поправку для версии МГТУ.


In [ ]:
# @title Файловый QC и кандидатный манифест
from record_qc import build_exp03_candidate_manifest

QC_MANIFEST_PATH, QC_MANIFEST = build_exp03_candidate_manifest(CONFIG_PATH)
QC_INCLUDED = [item for item in QC_MANIFEST["records"] if item["include"]]
QC_UNCLASSIFIED = [
    item for item in QC_MANIFEST["records"]
    if item["qc_status"] == "unclassified_not_included_pending_primary_protocol"
]
print("10.11 file_qc_status: pending_manual_review")
print("Основных включённых записей:", len(QC_INCLUDED))
print("Неклассифицированных уникальных записей:", len(QC_UNCLASSIFIED))
print("Кандидатный манифест:", QC_MANIFEST_PATH)


## Производный манифест записей

Последующие расчёты серии 40 принимают только обезличенный манифест
контроля качества (QC) по схеме
`schemas/record_manifest.schema.json`. Он должен фиксировать точный `record_id`,
SHA-256 исходной записи, добровольца, конфигурацию, монтаж, размер боковой сборки
и **фактически**, а не ожидаемо активные каналы. До ручного принятия такого
манифеста реальный расчёт серии 40 блокируется.
